# SQL Analysis — questions an analyst would ask of this hospital database

The first two notebooks explore the data with pandas and re-run the validation pipeline. This one asks business questions in **SQL**, against the same shipped CSVs (`app/data/`), loaded into an in-memory SQLite database, so it needs no database server.

Everything here is synthetic. The numbers are properties of the generator's settings, not findings about any hospital; the point is the queries: joins, window functions, CTEs, the polymorphic `staff_shifts` join, and a data-quality check expressed in SQL. SQLite dialect (`julianday`, `strftime`); the MySQL/MariaDB equivalents are `DATEDIFF` and `DATE_FORMAT`.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../app/data")
TABLES = [
    "departments", "payers", "procedures", "patients", "doctors", "nurses",
    "employees", "rooms", "admissions", "triage", "diagnoses",
    "patient_procedures", "medications", "medical_tests", "staff_shifts",
    "billing", "billing_line_items", "icd10_lookup",
]

con = sqlite3.connect(":memory:")
for t in TABLES:
    pd.read_csv(DATA_DIR / f"{t}.csv").to_sql(t, con, index=False)

def q(sql):
    """Run a query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con)

pd.set_option("display.width", 140)
print(f"SQLite {sqlite3.sqlite_version}; loaded {len(TABLES)} tables, "
      f"{q('SELECT COUNT(*) AS n FROM admissions').n[0]:,} admissions")

SQLite 3.45.1; loaded 18 tables, 12,000 admissions


## 1. Length of stay by department

A plain join and aggregate. `length_of_stay` only exists for discharged stays, so in-progress admissions are excluded rather than counted as zero-day stays.

In [2]:
q('''
    SELECT d.dep_name                       AS department,
           COUNT(*)                         AS discharges,
           ROUND(AVG(a.length_of_stay), 1)  AS avg_los_days,
           MAX(a.length_of_stay)            AS longest_stay
    FROM admissions a
    JOIN departments d USING (department_id)
    WHERE a.admission_status = 'Discharged'
    GROUP BY d.dep_name
    ORDER BY avg_los_days DESC
''')

,department,discharges,avg_los_days,longest_stay
0,OB/GYN,544,5.0,37.0
1,Hematology,517,4.9,31.0
2,Orthopedic,492,4.8,27.0
3,Neurology,487,4.8,27.0
4,Cardiology,461,4.8,32.0
5,Pediatrics,532,4.7,40.0
6,Intensive Care,950,4.7,31.0
7,General Surgery,455,4.7,34.0
8,Gastroenterology,481,4.7,32.0
9,Endocrinology,487,4.7,42.0


## 2. Repeat admissions within 30 days

A window function does what a self-join would: `LAG(discharge_date)` over each patient's stays in date order gives the previous discharge on the same row, so the gap is one subtraction. The CTE keeps the steps readable.

In real healthcare analytics this is the "30-day readmission" measure. Here readmissions happen by chance (the generator has no clinical link between stays), so this demonstrates the query, not a clinical result.

In [3]:
q('''
    WITH stays AS (
        SELECT patient_id, admission_id, admission_date,
               LAG(discharge_date) OVER (PARTITION BY patient_id ORDER BY admission_date) AS prev_discharge
        FROM admissions
        WHERE admission_status = 'Discharged'
    ),
    gaps AS (
        SELECT *, julianday(admission_date) - julianday(prev_discharge) AS gap_days
        FROM stays
        WHERE prev_discharge IS NOT NULL
    )
    SELECT COUNT(*)                                     AS repeat_admissions,
           SUM(gap_days <= 30)                          AS within_30_days,
           ROUND(100.0 * SUM(gap_days <= 30) / COUNT(*), 1) AS pct_within_30_days
    FROM gaps
''')

,repeat_admissions,within_30_days,pct_within_30_days
0,4743,1233,26.0


## 3. Billing by payer type

Three different percentages from one pass over `billing`, using `SUM(CASE WHEN ...)` for a conditional total. `insurance_covered / total_amount` is the coverage rate the generator sets per payer; "paid" is the share of billed dollars whose bill has reached `Paid` status.

In [4]:
q('''
    SELECT p.payer_type,
           COUNT(*)                                                           AS bills,
           ROUND(SUM(b.total_amount) / 1e6, 2)                                AS billed_millions,
           ROUND(100.0 * SUM(b.insurance_covered) / SUM(b.total_amount), 1)   AS pct_insurance_covered,
           ROUND(100.0 * SUM(CASE WHEN b.payment_status = 'Paid' THEN b.total_amount END)
                       / SUM(b.total_amount), 1)                              AS pct_billed_dollars_paid
    FROM billing b
    JOIN payers p USING (payer_id)
    GROUP BY p.payer_type
    ORDER BY billed_millions DESC
''')

,payer_type,bills,billed_millions,pct_insurance_covered,pct_billed_dollars_paid
0,Private Insurance,5183,70.38,76.2,49.5
1,Workers Compensation,1118,15.74,81.6,47.7
2,CHIP,1079,14.89,67.2,48.9
3,Medicare,1100,14.35,72.0,47.6
4,Self-Pay,1080,14.28,0.0,55.0
5,Medicaid,1058,14.15,64.2,49.6


## 4. The polymorphic join: `staff_shifts`

`staff_shifts.staff_id` points at a doctor, a nurse *or* an employee depending on `staff_type`, so there is no single foreign key and the database cannot enforce it. To query it, stack the three staff tables into one CTE with a `staff_type` column and join on **both** keys; joining on `staff_id` alone would work only because IDs happen not to collide.

Two checks on the result: shifts that matched no staff member at all (orphans, via `LEFT JOIN`) and shifts dated before the person's hire date. Both should be zero, and the validator's `shift_before_hire_or_wrong_dept` rule says the same.

In [5]:
q('''
    WITH staff AS (
        SELECT doctor_id   AS staff_id, 'Doctor'   AS staff_type, hire_date FROM doctors
        UNION ALL
        SELECT nurse_id,                'Nurse',                  hire_date FROM nurses
        UNION ALL
        SELECT employee_id,             'Employee',               hire_date FROM employees
    )
    SELECT sh.staff_type,
           COUNT(*)                                  AS shifts,
           ROUND(SUM(sh.hours_worked))               AS hours,
           SUM(st.staff_id IS NULL)                  AS orphan_shifts,
           SUM(sh.shift_date < st.hire_date)         AS shifts_before_hire
    FROM staff_shifts sh
    LEFT JOIN staff st
           ON st.staff_id = sh.staff_id AND st.staff_type = sh.staff_type
    GROUP BY sh.staff_type
''')

,staff_type,shifts,hours,orphan_shifts,shifts_before_hire
0,Doctor,4224,42232.0,0,0
1,Employee,6000,60052.0,0,0
2,Nurse,10800,107898.0,0,0


## 5. Top two diagnoses in each department

`ROW_NUMBER() OVER (PARTITION BY ...)` ranks within each group, and filtering on the rank in an outer query is the standard "top N per group" pattern. The ICD-10 code is joined to `icd10_lookup` so the output reads as names.

In [6]:
q('''
    WITH counts AS (
        SELECT d.dep_name AS department, l.diagnosis_name, COUNT(*) AS n
        FROM diagnoses dx
        JOIN admissions a   ON a.admission_id = dx.admission_id
        JOIN departments d  ON d.department_id = a.department_id
        JOIN icd10_lookup l ON l.icd10_code = dx.icd10_code
        GROUP BY d.dep_name, l.diagnosis_name
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY department ORDER BY n DESC, diagnosis_name) AS rnk
        FROM counts
    )
    SELECT department, rnk, diagnosis_name, n
    FROM ranked
    WHERE rnk <= 2
    ORDER BY department, rnk
''')

,department,rnk,diagnosis_name,n
0,Anesthesiology,1,"Chest pain, unspecified",424
1,Anesthesiology,2,Syncope and collapse,341
2,Cardiology,1,"Heart failure, unspecified",186
3,Cardiology,2,Essential (primary) hypertension,146
4,Emergency Medicine,1,"Sepsis, unspecified organism",109
5,Emergency Medicine,2,"Pneumonia, unspecified organism",104
6,Endocrinology,1,Type 2 diabetes mellitus without complications,446
7,Endocrinology,2,Type 2 diabetes mellitus with hyperglycemia,277
8,Gastroenterology,1,"Nausea with vomiting, unspecified",125
9,Gastroenterology,2,Calculus of gallbladder without cholecystitis,114


## 6. Where the money comes from

Share of billed dollars by line-item type, with the grand total computed by a scalar subquery so every row's percentage uses the same denominator.

In [7]:
q('''
    SELECT item_type,
           COUNT(*)                                    AS line_items,
           ROUND(SUM(total_cost) / 1e6, 2)             AS millions,
           ROUND(100.0 * SUM(total_cost)
                 / (SELECT SUM(total_cost) FROM billing_line_items), 1) AS pct_of_billed
    FROM billing_line_items
    GROUP BY item_type
    ORDER BY millions DESC
''')

,item_type,line_items,millions,pct_of_billed
0,Room,10618,85.45,59.4
1,Procedure,9600,32.03,22.3
2,Medication,19226,16.10,11.2
3,Lab,16112,6.48,4.5
4,Consultation,7434,3.72,2.6


## 7. A rolling average over time

Monthly admissions with a 3-month moving average. The window frame `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` averages the current month and the two before it; the CTE aggregates to months first, because window functions run after `GROUP BY`. Shown most recent first; scheduled and cancelled admissions are excluded because they never happened.

In [8]:
q('''
    WITH monthly AS (
        SELECT strftime('%Y-%m', admission_date) AS month, COUNT(*) AS admissions
        FROM admissions
        WHERE admission_status IN ('Admitted', 'Discharged')
        GROUP BY month
    )
    SELECT month, admissions,
           ROUND(AVG(admissions) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS rolling_3_month_avg
    FROM monthly
    ORDER BY month DESC
    LIMIT 12
''')

,month,admissions,rolling_3_month_avg
0,2026-10,407,751.0
1,2026-09,912,918.3
2,2026-08,934,914.0
3,2026-07,909,902.0
4,2026-06,899,896.7
5,2026-05,898,892.3
6,2026-04,893,870.0
7,2026-03,886,874.7
8,2026-02,831,877.3
9,2026-01,907,889.7


## 8. A data-quality rule written as SQL

The validator's clinical-coherence checks (Group 8) are plain SQL, so an analyst can re-run one by hand. This is `cancer_treatment_without_cancer_dx`: chemotherapy or radiation procedures (CPT 96413, 77373) on an admission with no cancer diagnosis (ICD-10 `C...`). `NOT EXISTS` is the anti-join; in SQLite a boolean sums as 0/1.

The generator now chooses treatments from each admission's own diagnoses, so the violation count should be zero. Before that change it was 680 of 839 such procedures in the earlier large dataset (see `docs/METHODOLOGY.md`).

In [9]:
q('''
    SELECT COUNT(*) AS cancer_treatments,
           SUM(NOT EXISTS (SELECT 1 FROM diagnoses d
                           WHERE d.admission_id = pp.admission_id
                             AND d.icd10_code LIKE 'C%')) AS without_cancer_diagnosis
    FROM patient_procedures pp
    JOIN procedures pr USING (procedure_id)
    WHERE pr.procedure_code IN ('96413', '77373')
''')

,cancer_treatments,without_cancer_diagnosis
0,693,0


Two more of the same kind, to see the rules from the patient's side: deliveries by patient sex, and insulin or metformin orders that lack a diabetes diagnosis. Both should show no violations.

In [10]:
print(q('''
    SELECT p.gender, COUNT(*) AS deliveries
    FROM patient_procedures pp
    JOIN procedures pr  USING (procedure_id)
    JOIN admissions a   ON a.admission_id = pp.admission_id
    JOIN patients p     ON p.patient_id = a.patient_id
    WHERE pr.procedure_code IN ('59400', '59510')
    GROUP BY p.gender
''').to_string(index=False))
print()
print(q('''
    SELECT COUNT(*) AS diabetes_drug_orders,
           SUM(NOT EXISTS (SELECT 1 FROM diagnoses d
                           WHERE d.admission_id = m.admission_id
                             AND (d.icd10_code LIKE 'E10%' OR d.icd10_code LIKE 'E11%'
                                  OR d.icd10_code LIKE 'E12%' OR d.icd10_code LIKE 'E13%'
                                  OR d.icd10_code LIKE 'E14%'))) AS without_diabetes_diagnosis
    FROM medications m
    WHERE m.drug_name LIKE 'Insulin%' OR m.drug_name = 'Metformin'
''').to_string(index=False))

gender  deliveries
     F          48



 diabetes_drug_orders  without_diabetes_diagnosis
                  786                           0


## Takeaways

- Window functions (`LAG`, `ROW_NUMBER`, a moving `AVG`) replace most self-joins and give the "previous row" or "top N per group" in one pass.
- A polymorphic reference like `staff_shifts.staff_id` needs a `UNION ALL` of the target tables and a two-column join, and an explicit orphan check, because the schema can't enforce it.
- A data-quality rule is just a query that should return zero; keeping the rules in SQL means the same logic runs in the pipeline and by hand.
- None of the figures above are findings: the data is synthetic and its distributions come from the generator's settings.